# Práctica: detección de fraude mediante métodos de ensembles

ESTUDIANTES: `SAID DANIEL HUARITA MOLLO`

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/fraud.jpg" style="width:600px;">

En esta práctica vamos a utilizar todos los conocimientos adquiridos en los ejercicios anteriores, con el objetivo de construir un detector automático de fraude en pagos con tarjeta.

## Instrucciones

A lo largo del notebook encontrarás celdas que debes rellenar con tu propio código. Sigue las instrucciones del notebook y presta atención a los siguientes iconos:

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
Deberás resolver el ejercicio escribiendo tu propio código o respuesta en la celda inmediatamente inferior.
    <b>La nota máxima que puede obtenerse con esta clase de ejercicios es de 7 sobre 10.</b>
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/exclamation.png" height="80" width="80" style="float: right;"/>

***
<font color=#2655ad>
Esto es una pista u observación de utilidad que puede ayudarte a resolver el ejercicio. Presta atención a estas pistas para comprender el ejercicio en mayor profundidad.
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/pro.png" height="80" width="80" style="float: right;"/>

***
<font color=#259b4c>
Este es un ejercicio avanzado que te puede ayudar a profundizar en el tema, y a conseguir una calificación más alta. <b>Resolviendo esta clase de ejercicios puedes llegar conseguir hasta 3 puntos sobre 10.</b> ¡Buena suerte!</font>

***

Para evitar problemas con imports o incompatibilidades se recomienda ejecutar este notebook en uno de los [entornos de Ensembles recomendados](https://github.com/albarji/teaching-environments-ensembles).

El siguiente código mostrará todas las gráficas en el propio notebook en lugar de generar una nueva ventana.

In [1]:
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

import re

## Carga y preparación de datos

Para este problema usaremos una versión reducida de los datos disponibles en la competición de Kaggle [IEEE-CIS Fraud Detection](https://www.kaggle.com/c/ieee-fraud-detection/data). Los datos a emplear están incluidos en la carpeta *data*, con ficheros separados para entrenamiento y test. Cada fichero incluye una gran cantidad de variables explicativas sobre la naturaleza de la operación con tarjeta realizada: puede encontrarse información sobre la naturaleza de estas variables en las [discusiones de la competición](https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203). La variable objetivo es `isFraud`, la cual toma el valor 1 para operaciones fraudulentas.

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
Carga los datos de entrenamiento y test en dos Pandas DataFrames con nombres <b>train</b> y <b>test</b>, respectivamente.
</font>

***

In [2]:
train = pd.read_csv('../data/train.csv',sep=',',index_col=0)
test = pd.read_csv('../data/test.csv',sep=',', index_col=0)

Los datos a utilizar presentan algunas problemáticas que debes resolver antes de pasar a la construcción de modelos:
* Valores de variables expresados como strings. scikit-learn no acepta variables expresadas de esta forma.
* Valores faltantes en muchas variables. Aunque varios modelos de scikit-learn pueden trabajar con valores faltantes sin problemas, otros modelos fallarán al encontrarlos.

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
Aplica las transformaciones que creas necesarias para tener los datos listos para el modelado. Ten en cuenta que cualquier proceso realizado debe aplicarse tanto a los datos de train como a los de test.
</font>

***

Revisamos las dimensiones de los datasets

In [3]:
print("Dimensiones del dataset de entrenamiento:",train.shape)
print("Dimensiones del dataset de prueba:",test.shape)

Dimensiones del dataset de entrenamiento: (60000, 394)
Dimensiones del dataset de prueba: (60000, 394)


Al tener tantas variables(394) utilizaremos la descripción del dataset de Kaggle, para abordar el problema de una manera adecuada.

- TransactionID: Identificador único de cada transacción.
- isFraud: Indicador de fraude.
- TransactionDT: Delta de tiempo dado una fecha y hora en especifico, (no es timestamp).
- TransactionAMT: Monto de la transacción en dolares americanos (USD).
- ProductCD: Código de producto, producto para cada transacción.
- card1 - card6: Información sobre el pago con tarjeta, tipo de tarjeta, categoría de la tarjeta, banco emisor, país, etc.
- addr: Domicilio del comprador, región y país de facturación.
- dist: Diferentes distancias, las columnas están renombradas para no revelar información sensible.
- P_ and (R__) emaildomain: Dominio de e-mail para comprador y quien recibe la compra.
- C1-C14: Contadores, como la cantidad de direcciones asociadas a la tarjeta de pago, etc.
- D1-D15: Diferencias de tiempo en días, entre ultima compra u otros, etc.
- M1-M9: Coincidencias, como nombres en tarjetas, dirección, etc.
- Vxxx: Variables que vienen de Vesta (entidad dueña de los datos), fueron creadas con un proceso de feature engineer, entre ellas ranking, contadores, y otras relaciones.


Podemos segmentar las variables que son similares en los siguientes grupos:
- cards: card1-card6
- C: C1-C14
- D: D1-D15
- M: M1-M9
- V: Vxxx Columns
- El resto de columnas lo guardaremos como columnas base (base_columns).

In [4]:
# Segmentamos las columnas que comienzan con card
card_pattern = re.compile(r'^(card)\d{1,2}$')
card_columns = [col for col in train.columns.tolist() if card_pattern.match(col)]

# Segmentamos las columnas que comienzan con C
c_pattern = re.compile(r'^C\d{1,2}$')
c_columns = [col for col in train.columns.tolist() if c_pattern.match(col)]

# Segmentamos las columnas que comienzan con D
d_pattern = re.compile(r'^D\d{1,2}$')
d_columns = [col for col in train.columns.tolist() if d_pattern.match(col)]

# Segmentamos las columnas que comienzan con M
m_pattern = re.compile(r'^M\d$')
m_columns = [col for col in train.columns.tolist() if m_pattern.match(col)]

# Segmentamos las columnas que comienzan con V
v_pattern = re.compile(r'^V\d{1,3}$')
v_columns = [col for col in train.columns.tolist() if v_pattern.match(col)]

# Mostramsos el número de columnas en cada segmento
print('Número de columnas card:',len(card_columns))
print('Número de columnas C:',len(c_columns))
print('Número de columnas D:',len(d_columns))
print('Número de columnas M:',len(m_columns))
print('Número de columnas V:',len(v_columns),'\n')

# Calculamos el número total de columnas segmentadas
segmented_columns = card_columns + c_columns + d_columns + m_columns + v_columns
print('Total columnas segmentadas:',len(segmented_columns),'\n')

# Calculamos el número total de columnas base
base_columns = [col for col in train.columns.tolist() if col not in segmented_columns]
print('Total columnas base:',len(base_columns),'\n')

print('Total columnas en el dataset:',len(train.columns.tolist()))

Número de columnas card: 6
Número de columnas C: 14
Número de columnas D: 15
Número de columnas M: 9
Número de columnas V: 339 

Total columnas segmentadas: 383 

Total columnas base: 11 

Total columnas en el dataset: 394


Ya que segmentamos las columnas podemos hacer el análisis segmento a segmento.

Verificación de duplicados, revisaremos si la columna TransactionID, solo tiene valores únicos.

In [5]:
print('Total transacciones únicas:', train['TransactionID'].nunique())

Total transacciones únicas: 60000


Revision del balance del dataset, ya que la variable objetivo es 'isFraud' procederemos a revisarla.

In [6]:
train['isFraud'].value_counts(normalize=True, dropna=False)

isFraud
0    0.833333
1    0.166667
Name: proportion, dtype: float64

Estamos ante un dataset desbalanceado, debemos tener en consideración durante el proceso de modelado.

Descartamos las siguientes columnas del segmento de columnas base, por las siguientes razones:
- 'isFraud': Es la variable objetivo.
- 'TransactionID': Columna de identificadores.
- 'TransactionDT': Realmente no es una fecha, asi que la dejamos afuera.

In [7]:
base_columns.remove('isFraud')
base_columns.remove('TransactionID')
base_columns.remove('TransactionDT')

### base_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [8]:
train[base_columns].dtypes

TransactionAmt    float64
ProductCD          object
addr1             float64
addr2             float64
dist1             float64
dist2             float64
P_emaildomain      object
R_emaildomain      object
dtype: object

- TransactionAmt: Es una variable numérica
- ProductCD: Es una variable categórica.
- addr1,addr2: Son variables categóricas que indican la dirección del comprador.
- dist1,dist2: Son variables numéricas.
- P_emaildomain,R_emaildomain: Variables categóricas.

#### Variables numéricas

Revisamos cuantos nas hay por columna.

In [9]:
train[['TransactionAmt','dist1','dist2']].isna().sum()

TransactionAmt        0
dist1             37328
dist2             55265
dtype: int64

- Ya que las distancias dependen de las direcciones, no se imputaran, se rellenara con -1, para que pueda ingresar al árbol de clasificación.
- Crearemos un indicador binario de missing para las distancias.

- Revisamos las distribuciones de las variables dist1, dist2, para verificar que si podemos imputar por -1.
- Aprovechamos de revisar la columna 'TransactionAmt' y ver si hay algo raro.

In [10]:
train[['TransactionAmt','dist1','dist2']].describe()

,TransactionAmt,dist1,dist2
count,60000.000000,22672.000000,4735.000000
mean,136.802754,126.066117,221.198099
std,230.267404,390.690397,490.389855
min,0.292000,0.000000,0.000000
25%,41.811000,3.000000,7.000000
50%,72.000000,9.000000,40.000000
75%,134.950000,25.000000,222.000000
max,5279.950000,10286.000000,6825.000000


- No existe -1 para las distancias, asi que todo ok.
- La columna 'TransactionAmt' no presenta comportamiento extraño, todos números mayores a 0.

Procedemos con las transformaciones para las distancias.

In [11]:
# Transformaciones en Train
# Creamos nuevas columnas para indicar si dist1 o dist2 tienen valores faltantes
train[['dist1_miss','dist2_miss']] = train[['dist1','dist2']].isna().astype(int)
# Rellenamos los valores faltantes en dist1 y dist2 con (-1)
train[['dist1_fix','dist2_fix']] = train[['dist1','dist2']].fillna(-1)

# Transformaciones en Test
# Creamos nuevas columnas para indicar si dist1 o dist2 tienen valores faltantes
test[['dist1_miss','dist2_miss']] = test[['dist1','dist2']].isna().astype(int)
# Rellenamos los valores faltantes en dist1 y dist2 con (-1)
test[['dist1_fix','dist2_fix']] = test[['dist1','dist2']].fillna(-1)

Las variables: 'TransactionAmt','dist1_miss','dist2_miss','dist1','dist2'

Ya están todas como tipo numérico, listas para ser utilizadas.

In [12]:
base_columns_num = ['TransactionAmt','dist1_miss','dist2_miss','dist1_fix','dist2_fix']
train[base_columns_num].dtypes

TransactionAmt    float64
dist1_miss          int64
dist2_miss          int64
dist1_fix         float64
dist2_fix         float64
dtype: object

Verificamos que ya no hayan nas

In [13]:
train[base_columns_num].isna().sum()

TransactionAmt    0
dist1_miss        0
dist2_miss        0
dist1_fix         0
dist2_fix         0
dtype: int64

#### Variables categóricas

Revisamos cuantos nas hay por columna.

In [14]:
train[['ProductCD','addr1','addr2','P_emaildomain','R_emaildomain']].isna().sum()

ProductCD            0
addr1             8857
addr2             8857
P_emaildomain     9320
R_emaildomain    43432
dtype: int64

- addr1, addr2: Al ser variables categóricas que indican la dirección del comprador no imputaremos y se mantendrán con categoría propia 'missing'.
- Los dominios de email al ser categóricos tampoco los tocaremos y los valores faltantes tendrán su propia categoría 'missing'.

Revisamos cuantos valores únicos hay por columna, incluyendo nas.

In [15]:
train[['ProductCD','addr1','addr2','P_emaildomain','R_emaildomain']].nunique(dropna=False)

ProductCD          5
addr1            148
addr2             35
P_emaildomain     60
R_emaildomain     56
dtype: int64

- Para la variable ProductCD, ya que tiene pocos valores únicos haremos one-hot encoding.
- Crearemos un flag cuando el dominio del comprador 'P_emaildomain' y el que recibe la compra 'R_emaildomain', son diferentes.
- Luego rellenaremos los valores perdidos con 'missing'.
- Haremos un 'frequency encoding', ya que los valores únicos por variable son muchos. Consiste en reemplazar cada categoría por el número (o proporción) de veces que aparece en el conjunto de datos.

In [16]:
# Codificación de la variable ProductCD utilizando OneHotEncoder
product_encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# Ajustamos el codificador en el conjunto de entrenamiento,
# transformamos tanto el conjunto de entrenamiento como el de prueba
train_prod_encoded = product_encoder.fit_transform(train[['ProductCD']])
test_prod_encoded = product_encoder.transform(test[['ProductCD']])

# Asignamos las nuevas columnas codificadas al DataFrame original en Train y Test
train[product_encoder.get_feature_names_out(['ProductCD']).tolist()]=train_prod_encoded
test[product_encoder.get_feature_names_out(['ProductCD']).tolist()]=test_prod_encoded

# Guardamos los nombres de las nuevas columnas codificadas
base_columns_cat = ['ProductCD_C', 'ProductCD_H', 'ProductCD_R', 'ProductCD_S', 'ProductCD_W']

In [17]:
# Generamos el flag cuando los dominios de correo electrónico de comprador o vendedor son iguales,
# Si son iguales, asignamos 1, de lo contrario, asignamos 0
# Asignamos en Train
train['email_domain_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(int)
# Asignamos en Test
test['email_domain_match'] = (test['P_emaildomain'] == test['R_emaildomain']).astype(int)

# Agregamos la nueva variable email_domain_match a la lista de columnas categóricas base
base_columns_cat.append('email_domain_match')
# Verificamos la distribución de la nueva variable email_domain_match en el conjunto de entrenamiento
train['email_domain_match'].value_counts(dropna=False, normalize=True)

email_domain_match
0    0.7838
1    0.2162
Name: proportion, dtype: float64

La coincidencia de dominio del email para comprador y quien recibe la compra se da el 21% de las veces.

Rellenamos valores perdidos con 'missing'

In [18]:
# Columnas categóricas en el segmento de columnas base
columns_cat = ['addr1','addr2','P_emaildomain','R_emaildomain']
columns_cat_fix = [col+'_fix' for col in columns_cat]

# Rellenamos los valores faltantes en las columnas categóricas con 'missing'
train[columns_cat_fix] = train[columns_cat].fillna('missing')
test[columns_cat_fix] = test[columns_cat].fillna('missing')

# Verificamos que ya no haya valores faltantes en las columnas
train[columns_cat_fix].isna().sum()

addr1_fix            0
addr2_fix            0
P_emaildomain_fix    0
R_emaildomain_fix    0
dtype: int64

Frequency encoding, para: 'addr1','addr2','P_emaildomain','R_emaildomain' en TRAIN

In [19]:
# Creamos un diccionario para almacenar los mapas de frecuencia de cada columna categórica
freq_maps_columns_cat = {}
for col in columns_cat_fix:
    freq_maps_columns_cat[col] = train[col].value_counts()

# Creamos nuevas columnas para cada columna categórica con la frecuencia de cada categoría
base_columns_cat_freq = []
for col in columns_cat_fix:
    colname_freq = col + '_freq'
    train[colname_freq] = train[col].map(freq_maps_columns_cat[col])
    base_columns_cat_freq.append(colname_freq)

train[base_columns_cat_freq].head()

,addr1_fix_freq,addr2_fix_freq,P_emaildomain_fix_freq,R_emaildomain_fix_freq
229713,3940,50584,46,43432
179605,1428,50584,795,43432
456372,49,94,24152,7831
362786,2025,50584,24152,43432
368636,4095,50584,146,7831


Frequency encoding, para: 'addr1','addr2','P_emaildomain','R_emaildomain' en TEST

In [20]:
# Rellenamos los valores faltantes en las columnas categóricas con 'missing'
test[columns_cat_fix] = test[columns_cat].fillna('missing')

# Creamos nuevas columnas para cada columna categórica con la frecuencia de cada categoría vista en train
for col in columns_cat_fix:
    colname_freq = col + '_freq'
    test[colname_freq] = test[col].map(freq_maps_columns_cat[col])

# Cuando mapeamos las frecuencias de las columnas categóricas en test, 
# es posible que algunas categorías no estén presentes en train,
# en estos casos tendran un valor NaN, como es la frecuencia imputamos a 0
test[base_columns_cat_freq] = test[base_columns_cat_freq].fillna(int(0))
test[base_columns_cat_freq] = test[base_columns_cat_freq].astype(int)

# Revisamos que no haya valores faltantes en las nuevas columnas de frecuencia en test
test[base_columns_cat_freq].isna().sum()

addr1_fix_freq            0
addr2_fix_freq            0
P_emaildomain_fix_freq    0
R_emaildomain_fix_freq    0
dtype: int64

Una vez finalizado el tratamiento a ambos dataset para las columnas 'base' podemos visualizar las columnas a utilizar posteriormente.

In [21]:
base_columns_final = base_columns_num + base_columns_cat_freq + base_columns_cat
train[base_columns_final].dtypes

TransactionAmt            float64
dist1_miss                  int64
dist2_miss                  int64
dist1_fix                 float64
dist2_fix                 float64
addr1_fix_freq              int64
addr2_fix_freq              int64
P_emaildomain_fix_freq      int64
R_emaildomain_fix_freq      int64
ProductCD_C               float64
ProductCD_H               float64
ProductCD_R               float64
ProductCD_S               float64
ProductCD_W               float64
email_domain_match          int64
dtype: object

In [22]:
train[base_columns_final].isna().sum()

TransactionAmt            0
dist1_miss                0
dist2_miss                0
dist1_fix                 0
dist2_fix                 0
addr1_fix_freq            0
addr2_fix_freq            0
P_emaildomain_fix_freq    0
R_emaildomain_fix_freq    0
ProductCD_C               0
ProductCD_H               0
ProductCD_R               0
ProductCD_S               0
ProductCD_W               0
email_domain_match        0
dtype: int64

Todas son numéricas y no tienen valores perdidos, procedemos con el siguiente grupo de variables.

### card_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [23]:
train[card_columns].dtypes

card1      int64
card2    float64
card3    float64
card4     object
card5    float64
card6     object
dtype: object

Según la descripción en Kaggle todas las variables son categóricas, como no todas son numéricas trabajaremos primero con las que ya son tipo numero y luego con las que no.

#### Variables numéricas

Revisamos cuantos nas hay por columna.

In [24]:
numeric_cols =train[card_columns].dtypes[train[card_columns].dtypes != 'object'].index.tolist()
train[numeric_cols].isna().sum()

card1      0
card2    977
card3    150
card5    454
dtype: int64

Revisamos valores únicos por variable, incluyendo nas.

In [25]:
train[numeric_cols].nunique(dropna=False)

card1    6059
card2     501
card3      78
card5      78
dtype: int64

Revisamos estadísticos básicos.

In [26]:
train[numeric_cols].describe()

,card1,card2,card3,card5
count,60000.000000,59023.000000,59850.000000,59546.000000
mean,9836.076333,363.037443,154.428271,198.214960
std,4888.927536,158.002592,12.797046,41.992689
min,1007.000000,100.000000,100.000000,100.000000
25%,6019.000000,214.000000,150.000000,166.000000
50%,9633.000000,361.000000,150.000000,226.000000
75%,14079.000000,512.000000,150.000000,226.000000
max,18390.000000,600.000000,229.000000,237.000000


Ya que los valores perdidos por columna son muy pocos, menos de 1000 observaciones. y se trabajara bajo el supuesto de que todas son categóricas.
- Se imputara con 'missing' los valores perdidos.
- Se creara un indicador de missing para estas columnas.
- Se realizara frequency encoding.

In [27]:
# Rellenamos los valores faltantes en las columnas categóricas con 'missing'
numeric_cols_fix = [col+'_fix' for col in numeric_cols]

train[numeric_cols_fix] = train[numeric_cols].fillna('missing')
test[numeric_cols_fix] = test[numeric_cols].fillna('missing')

# Verificamos que ya no haya valores faltantes en las columnas
train[numeric_cols_fix].isna().sum()

card1_fix    0
card2_fix    0
card3_fix    0
card5_fix    0
dtype: int64

Indicador de missing.

In [28]:
missing_flags = [col+'_missing' for col in numeric_cols]
train[missing_flags]=train[numeric_cols].isna().astype(int)
test[missing_flags]=test[numeric_cols].isna().astype(int)

Frequency encoding en Train.

In [29]:
# Creamos un diccionario para almacenar los mapas de frecuencia de cada columna categórica
freq_maps_columns_cat = {}
for col in numeric_cols_fix:
    freq_maps_columns_cat[col] = train[col].value_counts()

# Creamos nuevas columnas para cada columna categórica con la frecuencia de cada categoría
numeric_cols_freq = []
for col in numeric_cols_fix:
    colname_freq = col + '_freq'
    train[colname_freq] = train[col].map(freq_maps_columns_cat[col])
    numeric_cols_freq.append(colname_freq)

train[numeric_cols_freq].head()

,card1_fix_freq,card2_fix_freq,card3_fix_freq,card5_fix_freq
229713,25,51,50785,8364
179605,5,4132,50785,29471
456372,2,36,152,1751
362786,28,977,50785,5288
368636,7,715,50785,3275


Frequency encoding en Test.

In [30]:
# Rellenamos los valores faltantes en las columnas categóricas con 'missing'
test[numeric_cols_fix] = test[numeric_cols].fillna('missing')

# Creamos nuevas columnas para cada columna categórica con la frecuencia de cada categoría vista en train
for col in numeric_cols_fix:
    colname_freq = col + '_freq'
    test[colname_freq] = test[col].map(freq_maps_columns_cat[col])

# Cuando mapeamos las frecuencias de las columnas categóricas en test, 
# es posible que algunas categorías no estén presentes en train,
# en estos casos tendran un valor NaN, como es la frecuencia imputamos a 0
test[numeric_cols_freq] = test[numeric_cols_freq].fillna(0)
test[numeric_cols_freq] = test[numeric_cols_freq].astype(int)

# Revisamos que no haya valores faltantes en las nuevas columnas de frecuencia en test
test[numeric_cols_freq].isna().sum()

card1_fix_freq    0
card2_fix_freq    0
card3_fix_freq    0
card5_fix_freq    0
dtype: int64

Revisamos las columnas resultantes de este paso, tipo de dato.

In [31]:
cards_pt1 = numeric_cols_freq + missing_flags
train[cards_pt1].dtypes

card1_fix_freq    int64
card2_fix_freq    int64
card3_fix_freq    int64
card5_fix_freq    int64
card1_missing     int64
card2_missing     int64
card3_missing     int64
card5_missing     int64
dtype: object

Revisamos las columnas resultantes de este paso, si es que tiene valores perdidos.

In [32]:
train[cards_pt1].isna().sum()

card1_fix_freq    0
card2_fix_freq    0
card3_fix_freq    0
card5_fix_freq    0
card1_missing     0
card2_missing     0
card3_missing     0
card5_missing     0
dtype: int64

#### Variables categóricas

Revisamos cuantos nas hay por columna.

In [33]:
categoric_cols =[col for col in card_columns if col not in numeric_cols]
train[categoric_cols].isna().sum()

card4    154
card6    151
dtype: int64

Revisamos valores únicos por variable, incluyendo nas.

In [34]:
train[categoric_cols].nunique(dropna=False)

card4    5
card6    5
dtype: int64

Ya que los valores perdidos son pocos, y que la cantidad de categorías por cada variable es solo 5. Seguiremos con los siguientes pasos.

- Imputar con 'missing' los valores perdidos.
- Hacer one-hot encoding a ambas columnas.

In [35]:
# Rellenamos los valores faltantes en las columnas categóricas con 'missing'
categoric_cols_fix = [col+'_fix' for col in categoric_cols]

train[categoric_cols_fix] = train[categoric_cols].fillna('missing')
test[categoric_cols_fix] = test[categoric_cols].fillna('missing')

In [36]:
# Codificación de laa variables: card4, card6, utilizando OneHotEncoder
card_encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# Ajustamos el codificador en el conjunto de entrenamiento,
# transformamos tanto el conjunto de entrenamiento como el de prueba
train_card_encoded = card_encoder.fit_transform(train[['card4', 'card6']])
test_card_encoded = card_encoder.transform(test[['card4', 'card6']])

# Asignamos las nuevas columnas codificadas al DataFrame original en Train y Test
train[card_encoder.get_feature_names_out(['card4', 'card6']).tolist()]=train_card_encoded
test[card_encoder.get_feature_names_out(['card4', 'card6']).tolist()]=test_card_encoded

# Guardamos los nombres de las nuevas columnas codificadas
cards_pt2 = card_encoder.get_feature_names_out(['card4', 'card6']).tolist()

Revisamos la salida de este proceso.

In [37]:
cards_column_final = cards_pt1 + cards_pt2
train[cards_column_final].dtypes

card1_fix_freq              int64
card2_fix_freq              int64
card3_fix_freq              int64
card5_fix_freq              int64
card1_missing               int64
card2_missing               int64
card3_missing               int64
card5_missing               int64
card4_american express    float64
card4_discover            float64
card4_mastercard          float64
card4_visa                float64
card4_nan                 float64
card6_charge card         float64
card6_credit              float64
card6_debit               float64
card6_debit or credit     float64
card6_nan                 float64
dtype: object

In [38]:
train[cards_column_final].isna().sum()

card1_fix_freq            0
card2_fix_freq            0
card3_fix_freq            0
card5_fix_freq            0
card1_missing             0
card2_missing             0
card3_missing             0
card5_missing             0
card4_american express    0
card4_discover            0
card4_mastercard          0
card4_visa                0
card4_nan                 0
card6_charge card         0
card6_credit              0
card6_debit               0
card6_debit or credit     0
card6_nan                 0
dtype: int64

### C_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [39]:
train[c_columns].dtypes

C1     float64
C2     float64
C3     float64
C4     float64
C5     float64
C6     float64
C7     float64
C8     float64
C9     float64
C10    float64
C11    float64
C12    float64
C13    float64
C14    float64
dtype: object

Según la descripción son todos contadores, asi que coincide con el tipo de dato.

Revisamos cuantos nas hay por columna. En train y test.

In [40]:
train[c_columns].isna().sum()

C1     0
C2     0
C3     0
C4     0
C5     0
C6     0
C7     0
C8     0
C9     0
C10    0
C11    0
C12    0
C13    0
C14    0
dtype: int64

In [41]:
test[c_columns].isna().sum()

C1     0
C2     0
C3     0
C4     0
C5     0
C6     0
C7     0
C8     0
C9     0
C10    0
C11    0
C12    0
C13    0
C14    0
dtype: int64

No hay valores perdidos, revisamos brevemente estadísticos básicos.

In [42]:
train[c_columns].describe()

,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14
count,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000
mean,17.443133,19.903200,0.005550,5.796233,5.040733,10.357200,4.448117,7.779283,4.103500,7.645533,12.420483,6.536500,31.921500,8.686317
std,161.267703,188.880748,0.204173,82.598519,24.729080,84.606483,76.204287,116.967674,15.845896,117.175475,113.591280,107.074982,139.263003,57.163302
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
50%,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,2.000000,1.000000
75%,3.000000,3.000000,0.000000,1.000000,1.000000,2.000000,0.000000,1.000000,2.000000,1.000000,2.000000,0.000000,11.000000,2.000000
max,4668.000000,5624.000000,26.000000,2240.000000,302.000000,2240.000000,2242.000000,3317.000000,205.000000,3244.000000,3170.000000,3170.000000,2903.000000,1417.000000


No presenta valores extraños, todos son numéricos como deben ser, asi que no es necesario modificar nada.

### D_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [43]:
train[d_columns].dtypes

D1     float64
D2     float64
D3     float64
D4     float64
D5     float64
D6     float64
D7     float64
D8     float64
D9     float64
D10    float64
D11    float64
D12    float64
D13    float64
D14    float64
D15    float64
dtype: object

Según la descripción son todas diferencias, asi que coincide con el tipo de dato.

Revisamos cuantos nas hay por columna.

In [44]:
train[d_columns].isna().sum()

D1       128
D2     29790
D3     27432
D4     17308
D5     31119
D6     50604
D7     54402
D8     50381
D9     50381
D10     8341
D11    30264
D12    51318
D13    51842
D14    51728
D15     9664
dtype: int64

Tenemos muchos valores perdidos. Revisamos cuanto es en porcentaje.

In [45]:
train[d_columns].isna().sum()/train.shape[0]

D1     0.002133
D2     0.496500
D3     0.457200
D4     0.288467
D5     0.518650
D6     0.843400
D7     0.906700
D8     0.839683
D9     0.839683
D10    0.139017
D11    0.504400
D12    0.855300
D13    0.864033
D14    0.862133
D15    0.161067
dtype: float64

En algunos casos llegamos al 90% de missing data. A pesar de esto como puede ser una señal importante para detectar el fraude, crearemos variables para indicar la perdida de valor, veremos que hacer con los valores perdidos.

Revisamos estadísticos básicos.

In [46]:
train[d_columns].describe()

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15
count,59872.000000,30210.000000,32568.000000,42692.000000,28881.000000,9396.000000,5598.000000,9619.000000,9619.000000,51659.000000,29736.000000,8682.000000,8158.000000,8272.000000,50336.000000
mean,86.305936,159.711850,26.479489,130.480582,37.736609,59.637612,28.376742,114.016400,0.547536,114.036973,140.956080,48.826307,13.294680,55.781190,151.580300
std,152.093285,174.944797,60.419607,186.365499,84.743772,130.483709,82.154337,208.401920,0.324626,177.049608,183.596901,115.118126,55.054702,132.266399,198.270449
min,0.000000,0.000000,0.000000,-83.000000,0.000000,-83.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-83.000000,0.000000,-193.000000,-83.000000
25%,0.000000,20.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.875000,0.166666,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,86.000000,7.000000,17.000000,7.000000,0.000000,0.000000,17.708332,0.666666,7.000000,35.000000,0.000000,0.000000,0.000000,34.000000
75%,100.000000,255.000000,24.000000,226.250000,28.000000,25.000000,5.000000,129.791664,0.833333,169.000000,260.000000,14.000000,0.000000,4.000000,282.000000
max,640.000000,640.000000,649.000000,833.000000,736.000000,869.000000,706.000000,1295.833374,0.958333,832.000000,664.000000,648.000000,797.000000,847.000000,835.000000


Como tenemos varias columnas con valores negativos, revisamos a detalle estos para poder elegir el valor a imputar.

In [47]:
train[d_columns].describe().T['min'].sort_values(ascending=True)

D14   -193.0
D4     -83.0
D12    -83.0
D6     -83.0
D15    -83.0
D3       0.0
D2       0.0
D1       0.0
D8       0.0
D7       0.0
D5       0.0
D9       0.0
D11      0.0
D10      0.0
D13      0.0
Name: min, dtype: float64

El valor mas pequeño es -193, para mantener consistencia, imputaremos con -999 los valores faltantes.

In [48]:
# Creamos nuevas columnas para indicar si las columnas D tienen valores faltantes
d_missing_cols = [col+'_miss' for col in d_columns]
train[d_missing_cols] = train[d_columns].isna().astype(int)
test[d_missing_cols] = test[d_columns].isna().astype(int)

# Imputamos los valores faltantes en las columnas D con (-999)
d_columns_fix = [col+'_fix' for col in d_columns]
train[d_columns_fix] = train[d_columns].fillna(-999)
test[d_columns_fix] = test[d_columns].fillna(-999)

d_columns_final = d_columns_fix + d_missing_cols
train[d_columns_final].dtypes.value_counts()

float64    15
int64      15
Name: count, dtype: int64

Revisamos valores perdidos.

In [49]:
nas = train[d_columns_final].isna().sum()
nas[nas > 0]

Series([], dtype: int64)

Ahora todas las variables tienen son valores numéricos, y no tienen valores perdidos.

### M_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [50]:
train[m_columns].dtypes

M1    object
M2    object
M3    object
M4    object
M5    object
M6    object
M7    object
M8    object
M9    object
dtype: object

Según la descripción son todas coincidencias, debieran ser del tipo bool, revisamos.

Revisamos cuantos nas hay por columna.

In [51]:
train[m_columns].isna().sum()

M1    29444
M2    29444
M3    29444
M4    26830
M5    36004
M6    19708
M7    36629
M8    36629
M9    36629
dtype: int64

In [52]:
train[m_columns].isna().sum()/train.shape[0]

M1    0.490733
M2    0.490733
M3    0.490733
M4    0.447167
M5    0.600067
M6    0.328467
M7    0.610483
M8    0.610483
M9    0.610483
dtype: float64

Aproximadamente el 50% del data set tiene valores perdidos para estas columnas.

Revisamos cuantos valores únicos hay por columna incluyendo valores perdidos.

In [53]:
train[m_columns].nunique(dropna=False)

M1    3
M2    3
M3    3
M4    4
M5    3
M6    3
M7    3
M8    3
M9    3
dtype: int64

Tenemos en la mayoría 3 valores, solo un caso con 4, revisamos mas a detalle.

In [54]:
for col in m_columns:
    print(f"Valores únicos en {col}: {train[col].unique().tolist()}")

Valores únicos en M1: ['T', nan, 'F']
Valores únicos en M2: ['T', nan, 'F']
Valores únicos en M3: ['T', nan, 'F']
Valores únicos en M4: ['M0', nan, 'M2', 'M1']
Valores únicos en M5: ['F', nan, 'T']
Valores únicos en M6: ['F', 'T', nan]
Valores únicos en M7: ['F', nan, 'T']
Valores únicos en M8: ['F', nan, 'T']
Valores únicos en M9: ['F', nan, 'T']


Ya que en todas las variables con excepción de M4, tenemos T,F y na.

- Imputaremos con 'missing' cuando hayan valores faltantes.
- Utilizaremos label encoding en las que tienen solo 3 valores. (missing:-1,F:0,T:1)
- Utilizaremos one-hot encoding en la variable que tiene 4 valores posibles (M4).

In [55]:
# Imputamos los valores faltantes en las columnas M con 'missing'
m_columns_fix = [col+'_fix' for col in m_columns]

train[m_columns_fix] = train[m_columns].fillna('missing')
test[m_columns_fix] = test[m_columns].fillna('missing')

In [56]:
# creamos una lista de las columnas binarias en el segmento M, excluyendo M4
binary_cols = [col for col in m_columns if col!='M4']
# Creamos nuevas columnas binarias para cada columna en binary_cols,
# asignando 1 si el valor es 'T' y 0 si el valor es 'F' o 'missing'

# Diccionario para mapear los valores de las columnas binarias
binary_mapping = {'T': 1, 'F': 0, 'missing': -1}

encoded_cols = []
for col in binary_cols:
    encoded_col = col + '_bin'
    encoded_cols.append(encoded_col)
    # Aplicamos el mapeo a las columnas en train y test
    train[encoded_col] = train[col].map(binary_mapping)
    test[encoded_col] = test[col].map(binary_mapping)

In [57]:
# Hacemos one-hot encoding para la columna M4
m4_encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
train_m4_encoded = m4_encoder.fit_transform(train[['M4']])
test_m4_encoded = m4_encoder.transform(test[['M4']])

# Asignamos las nuevas columnas codificadas al DataFrame original en Train y Test
m4_cols = m4_encoder.get_feature_names_out(['M4']).tolist()
train[m4_cols] = train_m4_encoded
test[m4_cols] = test_m4_encoded

In [58]:
# Columnas finales del segmento M
m_columns_final = encoded_cols + m4_cols

Revisamos el tipo de dato.

In [59]:
train[m_columns_final].dtypes

M1_bin    float64
M2_bin    float64
M3_bin    float64
M5_bin    float64
M6_bin    float64
M7_bin    float64
M8_bin    float64
M9_bin    float64
M4_M0     float64
M4_M1     float64
M4_M2     float64
M4_nan    float64
dtype: object

Revisamos si hay valores perdidos.

In [60]:
train[m_columns_final].isna().sum()

M1_bin    29444
M2_bin    29444
M3_bin    29444
M5_bin    36004
M6_bin    19708
M7_bin    36629
M8_bin    36629
M9_bin    36629
M4_M0         0
M4_M1         0
M4_M2         0
M4_nan        0
dtype: int64

### V_columns

Revisamos los tipos de variables que tenemos, para poder agruparlas según el tipo y trabajar dependiendo del caso.

In [61]:
train[v_columns].dtypes.value_counts()

float64    339
Name: count, dtype: int64

Todas las columnas son numericas.

In [62]:
nas = train[v_columns].isna().sum()
len(nas[nas > 0])

339

Todas las columnas del segmento V, presentan valores perdidos.

In [63]:
(nas/train.shape[0]).describe()

count    339.000000
mean       0.419575
std        0.343797
min        0.000017
25%        0.002133
50%        0.504400
75%        0.741233
max        0.854817
dtype: float64

La cantidad de observaciones con valores perdido para las columnas del segmento V, va desde casi 0% hasta 85%, al ser muchas columnas, se imputara a la mediana. Ya que ir caso a caso es mucho más complejo, ademas de que no se tiene información acerca de lo que trata cada variable.

In [64]:
# Generamos nuevas columnas para las columnas V con la mediana imputada
v_columns_fix = [col+'_fix' for col in v_columns]

# Calculamos la mediana de cada columna en el conjunto de entrenamiento
median = train[v_columns].median()
train[v_columns_fix] = train[v_columns].fillna(median)
test[v_columns_fix] = test[v_columns].fillna(median)

C:\Users\Daniel\AppData\Local\Temp\ipykernel_18996\247979134.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[v_columns_fix] = train[v_columns].fillna(median)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_18996\247979134.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[v_columns_fix] = train[v_columns].fillna(median)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_18996\247979134.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

Revisamos nuevamente los valores perdidos.

In [65]:
nas = train[v_columns_fix].isna().sum()
len(nas[nas > 0])

0

### Selección final

Ya no hay valores perdidos para estas variables. generamos el dataset final.

In [66]:
final_columns = (base_columns_final 
                + cards_column_final
                + c_columns 
                + d_columns_final
                + m_columns_final
                + v_columns_fix
                )

train[final_columns].dtypes.value_counts()

float64    398
int64       30
Name: count, dtype: int64

Revisamos de que tengamos las mismas columnas en test.

In [67]:
test[final_columns].dtypes.value_counts()

float64    398
int64       30
Name: count, dtype: int64

Revisamos que las columnas en train y test, sean del mismo tipo.

In [69]:
match_columns = train[final_columns].dtypes == test[final_columns].dtypes
len(match_columns[match_columns == True])

428

Todo coincide ahora que tenemos un dataset limpio seguimos adelante.

## Midiendo el rendimiento de un detector de fraude

Las actividades fraudulentas son constantemente perseguidas, por lo que los defraudadores necesitan ser creativos e inventar nuevas formas de llevar a cabo sus fraudes. Además, afortunadamente, las operaciones fraudulentas son relativamente escasas, lo que nos lleva a contar con pocos casos positivos para entrenar el modelo. Dicho de otro modo, nos enfrentamos a un problema altamente desequilibrado, lo cual dificulta el entrenamiento del modelo, así como su evaluación.

Consideremos un modelo trivial que clasifica todos los casos como negativos (operaciones legítimas). Podemos simular las predicciones de este modelo creando un vector de predicciones de todo ceros:

In [ ]:
dumbpreds = [0] * len(test)

Midamos el porcentaje de acierto de este modelo sobre el conjunto de test

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(test["isFraud"], dumbpreds)

Deberías haber obtenido en torno a un 83% de acierto, ya que la gran mayoría de casos son negativos. A pesar de eso, ¡este modelo es totalmente inútil como detector de fraudes! Por tanto, necesitamos una métrica mejor.

Una métrica que funciona bien para problemas muy desbalanceados es el [área bajo la curva ROC](https://en.wikipedia.org/wiki/Receiver_operating_characteristic), o AUC. En scikit-learn esta métrica está disponible, y podemos probarla para comprobar que el rendimiento de este modelo es realmente malo

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(test["isFraud"], dumbpreds)

Un AUC del 50% indica que el modelo no es mejor que lanzar predicciones aleatoriamente. Si evaluáramos un modelo en el que las predicciones de probabilidad de la clase fraude fueran algo más altas para los casos realmente fraudulentos que para los casos legítimos, veríamos cómo el AUC produce mayores valores. El caso óptimo para esta métrica es un modelo en el que todos los casos de fraude son predichos con una mayor probabilidad de fraude que todos los casos de operaciones legítimas.

## Detector de fraude no supervisado

Dado que apenas tenemos datos de fraude, puede tener sentido empezar construyendo un modelo no supervisado.


<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
    Usando <b>solo los datos de train</b>, y sin emplear la variable <i>Class</i>, construye un modelo de tipo IsolationForest para detección de anomalías. Mide el rendimiento del modelo sobre el conjunto de test, usando la métrica AUC.
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/exclamation.png" height="80" width="80" style="float: right;"/>

***
<font color=#2655ad>
Ten en cuenta los siguientes puntos para construir un buen modelo:
    <ul>
        <li>Puedes estimar el parámetro <i>contamination</i> como la proporción de datos fraudulentos del conjunto de entrenamiento.</li>
        <li>Por defecto IsolationForest emplea pocos árboles. Asegúrate de probar con diferentes números de árboles.</li>
        <li>La métrica AUC necesita recibir las <b>probabilidades de clase fraude</b> para funcionar correctamente. No es posible obtener probabilidades de un IsolationForest, pero puedes hacer uso de su función <i>score_samples</i> para obtener scores (profundidades medias en el árbol), los cuales deben ser negados para obtener valores que pueden interpretarse como probabilidades de clase fraude (no normalizadas).</li>
    </ul>
    Si has entrenado el modelo correctamente, deberías obtener en torno a un <b>75% de AUC</b>.
</font>

***

In [ ]:
####### INSERT YOUR CODE HERE

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
    Crea una visualización mostrando el rendimiento de este modelo sobre los datos de test.
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/exclamation.png" height="80" width="80" style="float: right;"/>

***
<font color=#2655ad>
Sugerencia: utiliza la <a href=https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html>visualización ROC que se presenta en la documentación de scikit-learn</a>.
</font>

***

In [ ]:
####### INSERT YOUR CODE HERE

## Árbol de detección de fraude

Ahora vamos a comprobar si un método supervisado sencillo e interpretable, como es un árbol de decisión, puede ayudarnos a resolver este problema.

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
    Entrena un árbol de decisión sobre los datos de train, haciendo uso tanto de las variables explicativas como de la variable objetivo. Trata de que el árbol sea sencillo, de forma que puedas generar una visualización interpretable del mismo. Mide también los resultados del árbol en AUC sobre el conjunto de test. ¿Obtienes mejoras sobre el modelo no supervisado?
</font>

***

In [ ]:
####### INSERT YOUR CODE HERE

## Modelos supervisados de ensemble

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
    Vamos a comprobar cómo es posible mejorar los resultados de detección usando métodos de ensemble. Entrena <b>al menos 5 de los métodos de ensemble vistos en clase</b>, y mide sus resultados de AUC en el test. <b>Debes justificar todas las decisiones de diseño que tomes a la hora de elegir estos modelos, así como sus parámetros</b>.
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/exclamation.png" height="80" width="80" style="float: right;"/>

***
<font color=#2655ad>
Algunas sugerencias de modelos disponibles en scikit-learn que puedes utilizar:
    <ul>
        <li>Random Forest</li>
        <li>Extra Trees</li>
        <li>AdaBoost</li>
        <li>Gradient Boosting</li>
        <li>Histogram-based Gradient Boosting</li>
        <li>Bagging</li>
        <li>Voting</li>
        <li>Stacking</li>
    </ul>
No olvides que para medir correctamente el AUC, debes suministrar a esta métrica las <b>probabilidades de clase fraude</b> predichas por el modelo, las cuales puedes obtener mediante el método <i>predict_proba</i>.
    
Si has entrenado los modelos correctamente, tu mejor resultado debería ser de al menos un <b>92% de AUC</b> en test.
</font>

***

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/pro.png" height="80" width="80" style="float: right;"/>

***
<font color=#259b4c>
Adicionalmente, se valorará:
    <ul>
        <li>Utilizar más técnicas de ensembles, además de las 5 mínimas requeridas.</li>
        <li>Mejorar el AUC en test de tu modelo lo máximo posible.</li>
        <li>Utilizar <a href=https://catboost.ai/>CatBoost</a>, <a href=https://lightgbm.readthedocs.io/en/latest/>LightGBM</a> o <a href=https://auto.gluon.ai/>AutoGluon</a>, otras librerías de ensemble muy efectivas. Nótese que deberás instalar estas librerías en el entorno.</li>
    </ul>
</font>

***

In [ ]:
####### INSERT YOUR CODE HERE

## Visualización

<img src="https://albarji-labs-materials.s3-eu-west-1.amazonaws.com/question.png" height="80" width="80" style="float: right;"/>

***

<font color=#ad3e26>
    Crea una visualización mostrando el rendimiento de todos tus modelos supervisados sobre el conjunto de test, junto con el modelo no supervisado. La visualización debe mostrar claramente cuál es el modelo con mayor AUC.
</font>

***

In [ ]:
####### INSERT YOUR CODE HERE

In [ ]:
scores